In [ ]:
# Imports

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import Markdown
from IPython.display import Markdown, display
import pandas as pd

# SCI-KITLEARN

# LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             recall_score, precision_score, roc_auc_score)

# GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier


import joblib

#pyCaret
from pycaret.classification import setup, create_model, compare_models, plot_model, evaluate_model, predict_model, pull


In [ ]:
%matplotlib inline

Orginal data set:
https://www.kaggle.com/code/emineyetm/telco-customer-churn/notebook

Author of dataset:
https://www.kaggle.com/emineyetm

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
clean_df = pd.read_csv('data/telco_churn_clean.csv')

In [ ]:
clean_df.info()

## 1 pyCaret

### 1.1 Setup

In [ ]:
# setup(
#     data=clean_df,
#     target='Churn',
#     session_id=123,
# )

### 1.2 model selection

In [ ]:
# best_model = compare_models()

## Model Selection – Implications

#### 1. No clear winner

Since all models achieved similar performance, the choice of algorithm has a limited impact on overall predictive accuracy.

#### 2. Preference for simpler models

Given the negligible performance differences, a simpler model such as Logistic Regression is preferred due to its interpretability, stability, and lower computational cost.

#### 3. The importance of data > model

The results suggest that model performance is primarily constrained by the dataset, indicating that further improvements are more likely to come from feature engineering rather than more complex algorithms.

#### 4. Business consequence

The ability to interpret model outputs is crucial for business decision-making, making simpler models more practical despite similar predictive performance.

# 3 Sci-kitlearn operatoion

### 3.1. X/Y division

In [ ]:
X = clean_df.drop('Churn', axis=1)
y = clean_df['Churn']

### 3.2. Target encoding

In [ ]:
y = y.map({'Yes': 1, 'No': 0})

### 3.3. feature encoding
✔️ binary (Yes/No) 'Partner', 'Dependents', 'PaperlessBilling'

In [ ]:
binary_cols = ['Partner', 'Dependents', 'PaperlessBilling']

for col in binary_cols:
    X[col] = X[col].map({'Yes': 1, 'No': 0})

✔️ One-Hot else remains

In [ ]:
X = pd.get_dummies(X, drop_first=True)

### 3.4. Train / test split

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 3.5. First model (baseline)

In [ ]:


model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

### 3.6. Predyction

In [ ]:
y_pred = model.predict(X_test)

### 3.7. Assessment

In [ ]:
# from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

### The Logistic Regression model achieves an overall accuracy of 82%, but its ability to detect churn is limited, with a recall of only 58% for the churn class. This indicates that a significant portion of customers who are likely to leave are not correctly identified, which may limit the model's usefulness in a business context.

## 4. Improving actions:
1. class_weight (class_weight='balanced')

2. threshold tuning 0.5 → np. 0.3

1 - Probabilities

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]

2 - Seting Threshold

In [ ]:
# threshold = 0.3

# y_pred_custom = (y_proba >= threshold).astype(int)

In [ ]:
# print(classification_report(y_test, y_pred_custom))

In [ ]:
for t in [0.5, 0.4, 0.3, 0.2]:
    y_pred = (y_proba >= t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred))

## ROC curve

In [ ]:
# calculations
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

# plot
plt.figure(figsize=(7,5))

# ROC curve
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.2f})")

# diagonal line
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Model')

# descriptions
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

# legend
plt.legend(loc="lower right")

# siatka (ładniej wygląda)
plt.grid(True)

plt.show()

The ROC curve demonstrates that the model has strong discriminative ability, with an AUC of approximately 0.86, indicating good overall performance in distinguishing between churn and non-churn customers.

The curve represents the model's performance across all possible classification thresholds, highlighting the trade-off between true positive rate (recall) and false positive rate.

By adjusting the classification threshold, the model can be tuned to prioritize either higher recall (capturing more churn cases) or higher precision (reducing false positives).

In this analysis, thresholds in the range of 0.3–0.4 provide a reasonable balance, significantly improving churn detection while maintaining acceptable precision.

## 4. Final best model

#### Treshold = 0.4

#### - Why?

A threshold of 0.4 was selected as it provides a balanced trade-off between recall and precision. At this level, the model significantly improves the detection of churn cases compared to the default threshold (0.5), while keeping the number of false positives at an acceptable level. This balance is important from a business perspective, where identifying at-risk customers is valuable, but excessive false alerts may lead to inefficient resource allocation.

In [ ]:
threshold = 0.4

y_pred_custom_best = (y_proba >= threshold).astype(int)

In [ ]:
print(classification_report(y_test, y_pred_custom_best))

### Top Features Influencing Churn (Logistic Regression)

In [ ]:
# pobierz współczynniki
coefficients = model.coef_[0]

# przypisz do kolumn
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": coefficients
})

# sortuj
feature_importance = feature_importance.sort_values(by="importance", ascending=False)

# top 10
top = feature_importance.head(18)

# wykres
plt.figure(figsize=(8,5))
plt.barh(top["feature"], top["importance"])
plt.gca().invert_yaxis()
plt.title("Top Features Influencing Churn (Logistic Regression)")
plt.xlabel("Coefficient Value")
plt.show()

## 5. L1 model

### L1 regularization was applied to simplify the model by reducing the number of features, without impacting performance. The final model combines L1 regularization with a threshold of 0.4, resulting in improved churn detection and a more interpretable model.

In [ ]:
model_l1 = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    max_iter=1000,
    C=0.5  # we can change affter
)

model_l1.fit(X_train, y_train)

### how many features does the model use

In [ ]:
coef = model_l1.coef_[0]

display(Markdown(f"### Number of features: {len(coef)}"))
display(Markdown(f"### Non-zero features (in use): {np.sum(coef != 0)}"))

### Feature Importance (L1 Logistic Regression)

In [ ]:
feature_importance_l1 = pd.DataFrame({
    "feature": X.columns,
    "coef": coef
})

# only non-zero
selected = feature_importance_l1[feature_importance_l1["coef"] != 0]

# sort
selected = selected.sort_values(by="coef", ascending=False)

selected

### Plot for Feature Importance (L1 Logistic Regression)

In [ ]:
plt.figure(figsize=(8,5))
plt.barh(selected["feature"], selected["coef"])
plt.gca().invert_yaxis()
plt.title("Selected Features (L1 Logistic Regression)")
plt.show()

### Metric for Feature Importance (L1 Logistic Regression)

In [ ]:
y_pred_l1 = model_l1.predict(X_test)
print(classification_report(y_test, y_pred_l1))


The L1-regularized model achieves similar performance to the original Logistic Regression model, indicating that feature reduction did not negatively impact predictive quality. This suggests that many features were redundant and could be safely removed. However, the model still shows limited recall for churn (58%), meaning a significant portion of at-risk customers remains undetected.

# 6. Treshold 0.4 +L1

In [ ]:
y_proba_l1 = model_l1.predict_proba(X_test)[:, 1]

In [ ]:
threshold = 0.4

y_pred_l1_custom = (y_proba_l1 >= threshold).astype(int)

### Metric for Feature Importance (L1 Logistic Regression + Threshold 0.4)

In [ ]:
print(classification_report(y_test, y_pred_l1_custom))

The ROC AUC score for the L1-regularized Logistic Regression model is approximately 0.85, indicating good discriminative ability. This means the model is effective at distinguishing between customers who churn and those who remain. Importantly, this performance was achieved despite reducing the number of features, suggesting that the removed variables were not essential for prediction.

In [ ]:
auc_l1 = roc_auc_score(y_test, y_proba_l1)
print(f"AUC (L1 model): {auc_l1:.3f}")

## Confusion matrix

In [ ]:
print(confusion_matrix(y_test, y_pred_l1_custom))

In [ ]:
cm = confusion_matrix(y_test, y_pred_l1_custom)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix (L1 + threshold=0.4)")
plt.show()

### The model correctly identifies 252 churn cases while missing 121, achieving a recall of ~68%. It also produces 163 false positives, indicating a moderate trade-off between detection rate and operational cost.

## ROC curve with Threshold 0.4

In [ ]:
# obliczenia
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

# znajdź indeks dla threshold ~0.4
idx = np.argmin(np.abs(thresholds - 0.4))

# wykres
plt.figure(figsize=(7,5))

# krzywa ROC
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.2f})")

# linia losowa
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Model')

# 🔴 punkt dla threshold = 0.4
plt.scatter(fpr[idx], tpr[idx], color='red', s=80, label='Threshold = 0.4')

# opisy
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend(loc="lower right")
plt.grid(True)

plt.show()

## By adjusting the classification threshold to 0.4 and applying L1 regularization, the model improved its ability to detect churn cases, increasing recall from 58% to 68%. Although this resulted in a slight decrease in precision and overall accuracy, the model is now more effective in identifying at-risk customers, which is more valuable from a business perspective.

# 🔥 6. Cross-Validation

In [ ]:
model_cv = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    max_iter=1000,
    C=0.5
)

scores = cross_val_score(model_cv, X, y, cv=10, scoring='roc_auc')

print("AUC scores:", scores)
print("Mean AUC:", scores.mean())

#### Using 10-fold cross-validation, the model achieves AUC scores ranging from approximately 0.82 to 0.87, with a mean of 0.84. The relatively small variation across folds confirms that the model is stable and generalizes well, even under more rigorous validation.

# 7. Model Serialization

In [ ]:
final_model_bundle = {
    "model": model_l1,
    "threshold": 0.4,
    "features": X.columns.tolist()
}

joblib.dump(final_model_bundle, "clasyfication_model/churn_model_final.pkl")

## The final model is a Logistic Regression with L1 regularization, combined with a classification threshold of 0.4. This configuration provides a balance between model simplicity, interpretability, and improved churn detection performance.

# ==============
# GradientBoostingClassifier
# ==============

## 1. Bild GradientBoostingClassifier model

In [ ]:
gb_model = GradientBoostingClassifier(
    random_state=42
)

gb_model.fit(X_train, y_train)

#### Both Logistic Regression and Gradient Boosting were evaluated on the same train-test split to ensure a fair performance comparison.

### 1.1 Mean Predicted Churn Probability

In [ ]:
y_proba_gb = gb_model.predict_proba(X_test)[:, 1]
print(y_proba_gb.mean())

### 1.2 ROC AUC for Gradient Boosting Classifier

In [ ]:
print("ROC AUC:", roc_auc_score(y_test, y_proba_gb))

A ROC AUC score of 0.86 suggests that the model performs well at distinguishing customers likely to churn from those likely to stay.

### 1.3 Default Threshold Evaluation

In [ ]:
# threshold = 0.4

# y_pred_gb = (y_proba_gb >= threshold).astype(int)
# y_pred_gb = y_proba_gb.astype(int)
y_pred_gb = gb_model.predict(X_test)

print("Recall:", recall_score(y_test, y_pred_gb))
print("Precision:", precision_score(y_test, y_pred_gb))

The model correctly identified around 56% of actual churn cases, while approximately 68% of predicted churn cases were correct.

### 1.4  Threshold Tuning

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:

    y_pred_gb = (y_proba_gb >= threshold).astype(int)

    recall = recall_score(y_test, y_pred_gb)
    precision = precision_score(y_test, y_pred_gb)

    print(f"Threshold: {threshold}")
    print(f"Recall: {recall:.3f}")
    print(f"Precision: {precision:.3f}")
    print("-" * 30)

Different probability thresholds were evaluated to balance recall and precision. Lower thresholds increased recall but also produced a large number of false positives. Higher thresholds improved precision but reduced the model’s ability to detect actual churn cases.

A threshold of 0.4 provided the most balanced trade-off between recall and precision, maintaining solid churn detection performance while significantly reducing excessive false positive predictions.

### 1.3 Confusion matrix for every treshold

In [ ]:
for threshold in thresholds:

    y_pred_gb = (y_proba_gb >= threshold).astype(int)

    cm_gb = confusion_matrix(y_test, y_pred_gb)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm_gb)
    disp.plot()

    plt.title(f"Confusion Matrix - Threshold {threshold}")
    plt.show()

Lower thresholds improved churn detection but generated many false positives. Higher thresholds reduced false alarms at the cost of missing more actual churn cases.

Threshold 0.4 provided the most balanced business trade-off, maintaining strong churn detection performance while significantly reducing unnecessary churn predictions.

## 2 Tuning Gradient Boosting with learning rate

### 2.1 Tuning BG with learning_rate

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]

for lr in learning_rates:

    gb_model = GradientBoostingClassifier(
        learning_rate=lr,
        random_state=42
    )

    gb_model.fit(X_train, y_train)

    y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba_gb)

    print(f"Learning Rate: {lr}")
    print(f"ROC AUC: {auc:.4f}")
    print("-" * 30)

Different learning rates were evaluated to analyze how aggressively the Gradient Boosting model updated its predictions during training.

Very small learning rates resulted in slightly lower performance, likely because the model learned too slowly with the default number of estimators. Higher learning rates also reduced ROC AUC, suggesting that overly aggressive updates harmed generalization performance.

A learning rate of 0.05 achieved the best ROC AUC score and provided the most stable balance between learning speed and model generalization.

### 2.2 Choseing learning rate 0.05

In [ ]:
learning_rate = 0.05

gb_model = GradientBoostingClassifier(
    learning_rate=learning_rate,
    random_state=42
)

gb_model.fit(X_train, y_train)

y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba_gb)

print(f"Learning Rate: {learning_rate}")
print(f"ROC AUC: {auc:.4f}")



Lower learning rates produced better generalization performance, suggesting that slower and more incremental boosting updates helped the model avoid overfitting and improve class separation.

## 3 Gradient Boosting model tuning with n_estimators

In [ ]:
n_estimators_list = [50, 100, 200, 225, 300, 400]

for n in n_estimators_list:

    gb_model = GradientBoostingClassifier(
        learning_rate=0.05,
        n_estimators=n,
        random_state=42
    )

    gb_model.fit(X_train, y_train)

    y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba_gb)

    print(f"n_estimators: {n}")
    print(f"ROC AUC: {auc:.4f}")
    print("-" * 30)

Increasing the number of estimators improved model performance up to a certain point, after which ROC AUC began to gradually decline. The best performance was achieved with 100 estimators, suggesting that additional trees beyond this point provided diminishing returns and increased the risk of overfitting.

These results highlight the importance of balancing model complexity and generalization performance in boosting methods.

## 4. Threshold Tuning for Gradient Boosting (n_estimators and n estimators)

In [ ]:
gb_model = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=100,
    random_state=42
)

gb_model.fit(X_train, y_train)

y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

thresholds = [0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:

    y_pred_gb = (y_proba_gb >= threshold).astype(int)

    recall = recall_score(y_test, y_pred_gb)
    precision = precision_score(y_test, y_pred_gb)

    print(f"Threshold: {threshold}")
    print(f"Recall: {recall:.3f}")
    print(f"Precision: {precision:.3f}")
    print("-" * 30)

Threshold 0.4 provided the most balanced trade-off between recall and precision, maintaining strong churn detection while reducing excessive false positives.

## 5. Importance Feachurs for Gradient Boosting

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df

The feature importance analysis showed that customer tenure was the strongest predictor of churn, followed by internet service type, payment method, and contract duration. Customers with shorter tenure, fiber optic internet, and electronic check payments appeared to have higher churn risk. Long-term contracts were associated with lower churn probability, which aligns with expected business behavior.

Several features had near-zero importance, suggesting that the Gradient Boosting model relied primarily on a smaller subset of highly informative variables while effectively ignoring weak predictors. This demonstrates one advantage of tree-based ensemble methods: the ability to naturally reduce the influence of less useful features without explicit feature selection.

In [ ]:
import matplotlib.pyplot as plt

# top 15 feature
top_features = importance_df.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Gradient Boosting Feature Importance")

plt.gca().invert_yaxis()

plt.show()

## 6. Evaluating Whether Removing Low-Importance Features Improves Model Performance

In [ ]:
X_reduced = X.drop(
    columns=[
        "gender_Male",
        "Partner",
        "Dependents"
    ]
)

X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42
)

gb_model = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=100,
    random_state=42,
)

gb_model.fit(X_train_red, y_train_red)

y_proba_gb = gb_model.predict_proba(X_test_red)[:, 1]

auc_gb = roc_auc_score(y_test, y_proba_gb)


print(f"ROC AUC: {auc_gb:.4f}")

### Removing near-zero importance features had almost no effect on ROC AUC, indicating that the Gradient Boosting model was already effectively minimizing the influence of weak predictors.

### Because removing low-importance features did not meaningfully improve performance, the original feature set was retained in the final Gradient Boosting model.

## 7. Threshold Selection for the Tuned Model

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:

    y_pred_gb = (y_proba_gb >= threshold).astype(int)

    recall = recall_score(y_test, y_pred_gb)
    precision = precision_score(y_test, y_pred_gb)

    print(f"Threshold: {threshold}")
    print(f"Recall: {recall:.3f}")
    print(f"Precision: {precision:.3f}")
    print("-" * 30)

### 8. Gradient Boosting ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba_gb)

plt.figure(figsize=(8, 6))

plt.plot(fpr, tpr, label=f"GB ROC Curve (AUC = {auc:.3f})")

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Gradient Boosting ROC Curve")

plt.legend()

plt.show()

The ROC curve shows strong class separation performance, with the model achieving a ROC AUC score of approximately 0.86. The curve remains well above the random baseline, indicating that the model can effectively distinguish between churn and non-churn customers across different classification thresholds.

## 9. Cross Validation for Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=100,
    random_state=42
)

cv_scores = cross_val_score(
    gb_model,
    X,
    y,
    cv=10,
    scoring="roc_auc"
)

print("CV ROC AUC Scores:")
print(cv_scores)

print("\nMean ROC AUC:")
print(cv_scores.mean())

print("\nStd:")
print(cv_scores.std())

 The Gradient Boosting model demonstrated stable performance across 10-fold cross-validation, achieving a mean ROC AUC of approximately 0.85 with low variance between folds (std ≈ 0.013). This suggests that the model generalizes consistently across different subsets of the data and is not overly dependent on a single train-test split.

 Although individual fold scores varied slightly, the overall performance remained strong and stable, indicating a robust classification model for churn prediction.

## 10. Summary

The final Gradient Boosting model achieved strong classification performance on the Telco Customer Churn dataset, reaching a ROC AUC score of approximately 0.86. Model tuning focused on optimizing learning_rate and n_estimators, with the best results obtained using a learning rate of 0.05 and 100 estimators.

Threshold analysis demonstrated the trade-off between recall and precision across different decision boundaries. A threshold of 0.4 provided the most balanced business-oriented performance, maintaining solid churn detection while significantly reducing excessive false positive predictions.

Feature importance analysis identified customer tenure, internet service type, payment method, contract duration, and monthly charges as the strongest churn predictors. Removing near-zero importance features produced almost identical evaluation results, suggesting that the Gradient Boosting model was already effectively minimizing the influence of weak predictors.

Finally, 10-fold cross-validation confirmed stable model generalization performance, with consistently strong ROC AUC scores across different data splits.

## 11.Save the Gradiend Boosting model

In [ ]:
# Final model traing

gb_model = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=100,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_bundle = {
    "model": gb_model,
    "threshold": 0.4,
    "features": X_train.columns.tolist()
}

joblib.dump(
    gb_bundle,
    "clasyfication_model/gradient_boosting_model.pkl"
)

print("Gradient Boosting model saved.")

# ============
# Compare Logistic Regresion and Gradient Boosting
# ============

The final Gradient Boosting model was selected using a business-oriented evaluation approach focused on balancing churn detection performance and prediction reliability. After tuning learning_rate, n_estimators, and classification thresholds, the final configuration used a learning rate of 0.05, 100 estimators, and a threshold of 0.4.

This setup achieved strong ROC AUC performance while maintaining a balanced trade-off between recall and precision, making the model more practical for real-world churn prediction scenarios where both missed churn cases and excessive false positive alerts carry business costs.

Both the Logistic Regression and Gradient Boosting models were ultimately selected not only based on raw metrics, but also on their practical usefulness, interpretability, and ability to support business decision-making.

In [ ]:
print('ok')

In [ ]:
# Markdown("# PLAY")

In [ ]:
new_customer = pd.DataFrame([{
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 5,
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "Yes",
    "StreamingMovies": "No",
    "Contract": "Two year",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 100.0
}])

In [ ]:
# Markdown('## Liner Regression')

In [ ]:
# df_in = pd.read_csv('data/telco_churn_clean.csv')
# df_in.info()

In [ ]:
# bundle = joblib.load("clasyfication_model/churn_model_final.pkl")

# model = bundle["model"]
# threshold = bundle["threshold"]
# features = bundle["features"]

In [ ]:
# # maping
# binary_cols = ['Partner', 'Dependents', 'PaperlessBilling']
# for col in binary_cols:
#     new_customer[col] = new_customer[col].map({'Yes': 1, 'No': 0})

# # dummies
# X_new = pd.get_dummies(new_customer)

# # 🔥 KLUCZ
# X_new = X_new.reindex(columns=features, fill_value=0)

In [ ]:
# proba = model.predict_proba(X_new)[0][1]
# pred = (proba >= threshold).astype(int)

# print("Probability:", proba)
# print("Prediction:", pred)

In [ ]:
# Markdown("## Gradient Boosting - Final Evaluation")

In [ ]:
# LOAD GRADIENT BOOSTING MODEL

bundle_gb = joblib.load(
    "clasyfication_model/gradient_boosting_model.pkl"
)

model_gb = bundle_gb["model"]
threshold_gb = bundle_gb["threshold"]
features_gb = bundle_gb["features"]

In [ ]:
# PREPROCESS NEW CUSTOMER

new_customer_processed = new_customer.copy()

# binary mapping
binary_cols = ['Partner', 'Dependents', 'PaperlessBilling']

for col in binary_cols:
    new_customer_processed[col] = (
        new_customer_processed[col]
        .map({
            'Yes': 1,
            'No': 0
        })
    )

# one-hot encoding
X_new = pd.get_dummies(new_customer_processed)

# align columns with training features
X_new = X_new.reindex(
    columns=features_gb,
    fill_value=0
)

# remove potential NaN values
X_new = X_new.fillna(0)

In [ ]:
# PREDICTION

proba_gb = model_gb.predict_proba(X_new)[0][1]

pred_gb = (proba_gb >= threshold_gb).astype(int)

print("Probability:", proba_gb)
print("Prediction:", pred_gb)

---